# Logistic Regression Complete Project: Student Pass/Fail Prediction

## Project Goal

In this project, we will train a **Logistic Regression** model to predict whether a student will **Pass** or **Fail** based on study-related features.

This notebook is designed for teaching after Linear Regression.

### What students will learn

1. What Logistic Regression is
2. Difference between Regression and Classification
3. How to create/use a binary classification dataset
4. Exploratory Data Analysis
5. Feature selection
6. Train-test split
7. Feature scaling
8. Model training
9. Prediction
10. Probability prediction
11. Model evaluation
12. Confusion matrix
13. Accuracy, Precision, Recall, F1-score
14. ROC Curve and AUC
15. Predicting a new student's result
16. Saving and loading the trained model


# 1. What is Logistic Regression?

Logistic Regression is a **supervised learning classification algorithm**.

Even though its name contains the word "Regression", it is mainly used for **classification problems**.

## Example classification problems

- Pass or Fail
- Spam or Not Spam
- Customer Churn or Not Churn
- Loan Approved or Not Approved
- Disease or No Disease

## Linear Regression vs Logistic Regression

| Linear Regression | Logistic Regression |
|---|---|
| Predicts continuous values | Predicts categories/classes |
| Example: house price | Example: pass/fail |
| Output can be any number | Output is probability between 0 and 1 |
| Used for regression | Used for classification |

## In this project

Our target column will be:

```text
passed
```

Where:

```text
1 = Pass
0 = Fail
```


# 2. Import Required Libraries

We will use:

- `pandas` for data handling
- `numpy` for numerical operations
- `matplotlib` and `seaborn` for visualization
- `scikit-learn` for machine learning
- `joblib` for saving/loading the trained model


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score
)

import joblib

plt.rcParams["figure.figsize"] = (8, 5)

print("Libraries imported successfully!")

# 3. Create a Student Pass/Fail Dataset

For teaching, we will create a simple dataset.

Each row represents one student.

## Features

| Feature | Meaning |
|---|---|
| study_hours | How many hours the student studies daily |
| attendance | Attendance percentage |
| previous_score | Previous exam score |
| assignments_completed | Number of assignments completed |
| sleep_hours | Average sleep hours |
| internet_usage_hours | Daily non-study internet usage |

## Target

| Target | Meaning |
|---|---|
| passed | 1 means Pass, 0 means Fail |

In a real project, you may load data using:

```python
df = pd.read_csv("student_data.csv")
```

Here, we generate data so the notebook works without downloading any external file.


In [ ]:
np.random.seed(42)

num_students = 300

study_hours = np.random.normal(loc=4.5, scale=1.8, size=num_students)
study_hours = np.clip(study_hours, 0.5, 10)

attendance = np.random.normal(loc=75, scale=15, size=num_students)
attendance = np.clip(attendance, 30, 100)

previous_score = np.random.normal(loc=65, scale=15, size=num_students)
previous_score = np.clip(previous_score, 20, 100)

assignments_completed = np.random.randint(0, 11, size=num_students)

sleep_hours = np.random.normal(loc=7, scale=1.2, size=num_students)
sleep_hours = np.clip(sleep_hours, 3, 10)

internet_usage_hours = np.random.normal(loc=4, scale=1.5, size=num_students)
internet_usage_hours = np.clip(internet_usage_hours, 0.5, 9)

# Create a hidden score that decides pass/fail.
# Positive factors: study_hours, attendance, previous_score, assignments_completed, sleep_hours
# Negative factor: internet_usage_hours
pass_score = (
    study_hours * 2.2
    + attendance * 0.08
    + previous_score * 0.10
    + assignments_completed * 0.8
    + sleep_hours * 0.5
    - internet_usage_hours * 0.9
    + np.random.normal(0, 2, num_students)
)

# Convert score into binary result
passed = (pass_score > np.median(pass_score)).astype(int)

df = pd.DataFrame({
    "study_hours": study_hours.round(2),
    "attendance": attendance.round(2),
    "previous_score": previous_score.round(2),
    "assignments_completed": assignments_completed,
    "sleep_hours": sleep_hours.round(2),
    "internet_usage_hours": internet_usage_hours.round(2),
    "passed": passed
})

df.head()

# 4. Understand the Dataset

Before training any Machine Learning model, always understand the dataset.

We will check:

1. First 5 rows
2. Shape of the dataset
3. Column names
4. Data types
5. Summary statistics


In [ ]:
print("Dataset shape:", df.shape)
print("
Column names:")
print(df.columns.tolist())

print("
Dataset information:")
df.info()

In [ ]:
df.describe()

# 5. Check Target Distribution

For classification problems, we should check whether the target classes are balanced or imbalanced.

A balanced dataset means both classes have similar counts.

Example:

```text
Pass = 150
Fail = 150
```

An imbalanced dataset means one class is much larger than the other.

Example:

```text
Pass = 280
Fail = 20
```

Class imbalance can make accuracy misleading.


In [ ]:
df["passed"].value_counts()

In [ ]:
sns.countplot(data=df, x="passed")
plt.title("Pass vs Fail Count")
plt.xlabel("Passed: 0 = Fail, 1 = Pass")
plt.ylabel("Number of Students")
plt.show()

# 6. Exploratory Data Analysis

Now we will explore how different features relate to the target column.

This helps students understand the data before applying Machine Learning.


In [ ]:
sns.scatterplot(data=df, x="study_hours", y="previous_score", hue="passed")
plt.title("Study Hours vs Previous Score")
plt.xlabel("Study Hours")
plt.ylabel("Previous Score")
plt.show()

In [ ]:
sns.boxplot(data=df, x="passed", y="study_hours")
plt.title("Study Hours Distribution by Result")
plt.xlabel("Passed: 0 = Fail, 1 = Pass")
plt.ylabel("Study Hours")
plt.show()

In [ ]:
sns.boxplot(data=df, x="passed", y="attendance")
plt.title("Attendance Distribution by Result")
plt.xlabel("Passed: 0 = Fail, 1 = Pass")
plt.ylabel("Attendance")
plt.show()

In [ ]:
correlation = df.corr(numeric_only=True)

sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

# 7. Check Missing Values

Missing values can create errors during model training.

Common ways to handle missing values:

1. Remove rows
2. Fill with mean/median
3. Fill with mode
4. Use advanced imputation

Our generated dataset does not contain missing values, but we still check because this is an important ML habit.


In [ ]:
df.isnull().sum()

# 8. Select Features and Target

In Machine Learning:

- `X` means features/input columns
- `y` means target/output column

Here:

```text
X = student study-related features
y = passed
```


In [ ]:
X = df.drop("passed", axis=1)
y = df["passed"]

print("Feature columns:")
print(X.columns.tolist())

print("
Target column:")
print("passed")

# 9. Train-Test Split

We split the dataset into two parts:

## Training Data

Used to train the model.

## Testing Data

Used to check how well the model performs on unseen data.

A common split is:

```text
80% training
20% testing
```

We use `stratify=y` to keep the pass/fail ratio similar in both training and testing data.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

# 10. Feature Scaling

Logistic Regression performs better when features are on a similar scale.

Example:

- `attendance` ranges from 30 to 100
- `study_hours` ranges from 0.5 to 10

Scaling brings features to a common scale.

## Very important rule

Use:

```python
fit_transform()
```

on training data.

Use:

```python
transform()
```

on testing data.

Why?

Because the model should not learn information from the test data before evaluation.


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed!")

# 11. Train Logistic Regression Model

Now we train the Logistic Regression model.

The model will learn patterns from the training data.

In simple words, it learns:

```text
What type of student usually passes?
What type of student usually fails?
```


In [ ]:
model = LogisticRegression(random_state=42)

model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully!")

# 12. Make Predictions

Now we use the trained model to predict results for the test data.

The model will output:

```text
0 = Fail
1 = Pass
```


In [ ]:
y_pred = model.predict(X_test_scaled)

print("First 20 predictions:")
print(y_pred[:20])

print("
First 20 actual values:")
print(y_test.values[:20])

# 13. Predict Probabilities

Logistic Regression can also predict probabilities.

For example:

```text
Fail probability = 0.20
Pass probability = 0.80
```

By default, if pass probability is greater than or equal to 0.5, the model predicts Pass.


In [ ]:
y_prob = model.predict_proba(X_test_scaled)

probability_df = pd.DataFrame({
    "Fail_Probability": y_prob[:, 0],
    "Pass_Probability": y_prob[:, 1],
    "Predicted_Class": y_pred,
    "Actual_Class": y_test.values
})

probability_df.head(10)

# 14. Evaluate the Model

For classification, we use different evaluation metrics.

## Accuracy

Out of all predictions, how many were correct?

## Precision

When the model predicted Pass, how often was it actually Pass?

## Recall

Out of all actual Pass students, how many did the model correctly find?

## F1-score

A balance between Precision and Recall.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1-score:", round(f1, 4))

# 15. Confusion Matrix

A confusion matrix shows correct and incorrect predictions in detail.

For binary classification:

|  | Predicted Fail | Predicted Pass |
|---|---|---|
| Actual Fail | True Negative | False Positive |
| Actual Pass | False Negative | True Positive |

This is one of the most important concepts in classification.


In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# 16. Classification Report

The classification report gives precision, recall, and F1-score for each class.

Class 0 = Fail  
Class 1 = Pass


In [ ]:
print(classification_report(y_test, y_pred, target_names=["Fail", "Pass"]))

# 17. ROC Curve and AUC Score

ROC curve shows how well the model separates the two classes.

AUC score ranges from 0 to 1.

| AUC Score | Meaning |
|---|---|
| 0.5 | Random guessing |
| 0.7 - 0.8 | Acceptable |
| 0.8 - 0.9 | Good |
| 0.9 - 1.0 | Excellent |

For this beginner project, the goal is not just high score. The goal is to understand the full classification workflow.


In [ ]:
pass_probabilities = y_prob[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, pass_probabilities)
auc_score = roc_auc_score(y_test, pass_probabilities)

plt.plot(fpr, tpr, label=f"AUC = {auc_score:.2f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Guess")
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.show()

print("AUC Score:", round(auc_score, 4))

# 18. Model Coefficients

Logistic Regression learns coefficients for each feature.

Positive coefficient means the feature increases the chance of passing.

Negative coefficient means the feature decreases the chance of passing.

This is useful because Logistic Regression is more interpretable than many advanced models.


In [ ]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
}).sort_values(by="Coefficient", ascending=False)

coefficients

In [ ]:
sns.barplot(data=coefficients, x="Coefficient", y="Feature")
plt.title("Logistic Regression Feature Coefficients")
plt.xlabel("Coefficient Value")
plt.ylabel("Feature")
plt.show()

# 19. Predict Result for a New Student

Now we will test the model on a new student.

Important:

The new student's data must have the same columns as the training data.

Also, we must scale the new data using the same scaler.


In [ ]:
new_student = pd.DataFrame({
    "study_hours": [6],
    "attendance": [85],
    "previous_score": [72],
    "assignments_completed": [8],
    "sleep_hours": [7],
    "internet_usage_hours": [2]
})

new_student_scaled = scaler.transform(new_student)

new_prediction = model.predict(new_student_scaled)
new_probability = model.predict_proba(new_student_scaled)

print("Prediction:", "Pass" if new_prediction[0] == 1 else "Fail")
print("Fail Probability:", round(new_probability[0][0], 4))
print("Pass Probability:", round(new_probability[0][1], 4))

# 20. Change the Classification Threshold

By default, Logistic Regression uses threshold 0.5.

```text
If Pass Probability >= 0.5 → Pass
If Pass Probability < 0.5 → Fail
```

Sometimes, we change threshold depending on business needs.

Example:

If failing a weak student is risky, we may increase the threshold for Pass.

Let us test threshold 0.6.


In [ ]:
custom_threshold = 0.6

y_pred_custom = (pass_probabilities >= custom_threshold).astype(int)

print("Metrics with threshold 0.6")
print("Accuracy:", round(accuracy_score(y_test, y_pred_custom), 4))
print("Precision:", round(precision_score(y_test, y_pred_custom), 4))
print("Recall:", round(recall_score(y_test, y_pred_custom), 4))
print("F1-score:", round(f1_score(y_test, y_pred_custom), 4))

print("
Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_custom))

# 21. Save the Model and Scaler

When we deploy a Machine Learning model, we need to save:

1. The trained model
2. The scaler

Why save the scaler?

Because new data must be scaled in the same way as training data.


In [ ]:
joblib.dump(model, "logistic_regression_student_model.pkl")
joblib.dump(scaler, "student_scaler.pkl")

print("Model saved as logistic_regression_student_model.pkl")
print("Scaler saved as student_scaler.pkl")

# 22. Load the Saved Model

Now we load the model and scaler again to make sure they work.


In [ ]:
loaded_model = joblib.load("logistic_regression_student_model.pkl")
loaded_scaler = joblib.load("student_scaler.pkl")

loaded_student_scaled = loaded_scaler.transform(new_student)
loaded_prediction = loaded_model.predict(loaded_student_scaled)

print("Loaded model prediction:", "Pass" if loaded_prediction[0] == 1 else "Fail")

# 23. Complete Teaching Summary

## Logistic Regression Complete Workflow

1. Understand the classification problem
2. Load or create the dataset
3. Explore the data
4. Check target distribution
5. Check missing values
6. Select features and target
7. Split into training and testing data
8. Scale features
9. Train Logistic Regression model
10. Make predictions
11. Predict probabilities
12. Evaluate using classification metrics
13. Analyze confusion matrix
14. Analyze ROC curve and AUC score
15. Interpret coefficients
16. Predict a new sample
17. Save and load model

## Key concepts students must understand

- Logistic Regression is for classification
- Output is probability
- Default threshold is 0.5
- Accuracy is not always enough
- Precision and Recall are very important
- Confusion matrix is essential
- Scaling is important for Logistic Regression
- Use `fit_transform` on training data and `transform` on testing data


# 24. Student Assignment

Ask students to complete these tasks:

## Basic Tasks

1. Change the new student values and check prediction
2. Try threshold 0.4, 0.5, 0.6, and 0.7
3. Compare accuracy, precision, recall, and F1-score
4. Add a new feature called `practice_tests_taken`
5. Retrain the model

## Intermediate Tasks

1. Create another dataset for customer churn
2. Train Logistic Regression on that dataset
3. Create confusion matrix
4. Explain false positives and false negatives

## Teacher Discussion Questions

1. Why is Logistic Regression used for classification?
2. Why do we scale features?
3. Why is accuracy not always enough?
4. What is the difference between precision and recall?
5. What happens when we increase the classification threshold?
